# 🎯 Prompt Engineering - TripTrove RAG

Notebook untuk eksperimen dengan prompt engineering dan optimasi respons AI.

## Tujuan:
- Test berbagai prompt templates
- Optimize system instructions
- Experiment dengan temperature settings
- Compare response quality

In [ ]:
# Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / 'src'
sys.path.insert(0, str(src_path))

print(f"✅ Path configured: {src_path}")

In [ ]:
# Imports
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time

print("✅ Libraries imported!")

## 1. Initialize LLM with Different Settings

In [ ]:
# Create LLMs with different temperatures
llm_creative = ChatOllama(model="llama3.1", temperature=0.7)
llm_balanced = ChatOllama(model="llama3.1", temperature=0.3)
llm_precise = ChatOllama(model="llama3.1", temperature=0.1)

print("✅ LLMs initialized with different temperatures")

## 2. Test Different Prompt Templates

In [ ]:
# Define prompt templates
templates = {
    'simple': """
Context: {context}

Question: {question}

Answer:
""",
    
    'detailed': """
Anda adalah asisten AI untuk TripTrove, platform booking tour travel.

Context Information:
{context}

User Question: {question}

Instructions:
- Berikan jawaban yang akurat berdasarkan context
- Gunakan bahasa yang ramah dan profesional
- Jika tidak tahu, katakan dengan jujur

Answer:
""",
    
    'structured': """
System: Anda adalah TripTrove AI Assistant yang membantu customer menemukan paket tour.

Available Information:
{context}

Customer Question:
{question}

Response Guidelines:
1. Analyze the question type (price inquiry, recommendation, policy, general)
2. Extract relevant information from context
3. Provide clear, actionable answer
4. Include specific details (prices, dates, features)
5. End with helpful suggestion if appropriate

Your Response:
"""
}

print(f"✅ Created {len(templates)} prompt templates")

## 3. Compare Template Performance

In [ ]:
# Test context and question
test_context = """
Tour Package: Bali Paradise 3D2N
Price: Rp 2,500,000
Discount: 15%
Final Price: Rp 2,125,000
Duration: 3 days 2 nights
Category: Beach & Relaxation
Rating: 4.8/5.0
Includes: Hotel, breakfast, airport transfer, tour guide
"""

test_question = "Berapa harga paket tour ke Bali?"

print("Test setup ready!")

In [ ]:
# Test each template
results = {}

for template_name, template_text in templates.items():
    print(f"\n{'='*80}")
    print(f"Testing: {template_name.upper()}")
    print(f"{'='*80}")
    
    prompt = ChatPromptTemplate.from_template(template_text)
    chain = prompt | llm_balanced | StrOutputParser()
    
    start = time.time()
    response = chain.invoke({
        'context': test_context,
        'question': test_question
    })
    elapsed = time.time() - start
    
    print(f"\nResponse:\n{response}")
    print(f"\nTime: {elapsed:.2f}s")
    
    results[template_name] = {
        'response': response,
        'time': elapsed
    }

## 4. Test Temperature Effects

In [ ]:
# Use best template with different temperatures
best_template = templates['structured']
prompt = ChatPromptTemplate.from_template(best_template)

llms = {
    'Creative (0.7)': llm_creative,
    'Balanced (0.3)': llm_balanced,
    'Precise (0.1)': llm_precise
}

for name, llm in llms.items():
    print(f"\n{'='*80}")
    print(f"Temperature: {name}")
    print(f"{'='*80}")
    
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({
        'context': test_context,
        'question': test_question
    })
    
    print(f"\n{response}")

## 5. Test Query-Specific Instructions

In [ ]:
# Query-specific prompts
query_prompts = {
    'price': """
You are answering a PRICE INQUIRY.

Context: {context}
Question: {question}

Instructions:
- State the exact price clearly
- Mention any discounts
- Show final price after discount
- List what's included

Answer:
""",
    
    'recommendation': """
You are providing a RECOMMENDATION.

Context: {context}
Question: {question}

Instructions:
- Understand customer needs
- Suggest best matching tour
- Explain why it's suitable
- Mention key features

Answer:
""",
    
    'policy': """
You are answering a POLICY QUESTION.

Context: {context}
Question: {question}

Instructions:
- Be precise and accurate
- Quote exact terms if available
- Explain clearly
- Mention important conditions

Answer:
"""
}

print(f"✅ Created {len(query_prompts)} query-specific prompts")

In [ ]:
# Test query-specific prompts
test_cases = [
    ('price', "Berapa harga paket tour ke Bali?"),
    ('recommendation', "Rekomendasi paket tour untuk honeymoon?"),
    ('policy', "Apa kebijakan pembatalan TripTrove?")
]

for qtype, question in test_cases:
    print(f"\n{'='*80}")
    print(f"Query Type: {qtype.upper()}")
    print(f"Question: {question}")
    print(f"{'='*80}")
    
    prompt = ChatPromptTemplate.from_template(query_prompts[qtype])
    chain = prompt | llm_balanced | StrOutputParser()
    
    response = chain.invoke({
        'context': test_context,
        'question': question
    })
    
    print(f"\nResponse:\n{response}")

## 6. Interactive Prompt Testing

In [ ]:
# Custom prompt testing
custom_prompt = input("Enter your custom prompt template (use {context} and {question}): ")
test_q = input("Enter test question: ")

if custom_prompt and test_q:
    prompt = ChatPromptTemplate.from_template(custom_prompt)
    chain = prompt | llm_balanced | StrOutputParser()
    
    response = chain.invoke({
        'context': test_context,
        'question': test_q
    })
    
    print(f"\n{'='*80}")
    print("Response:")
    print(f"{'='*80}")
    print(response)

## 📝 Best Practices

Dari eksperimen di atas, catat:
1. Template mana yang menghasilkan respons terbaik?
2. Temperature berapa yang paling optimal?
3. Apakah query-specific instructions membantu?
4. Bagaimana cara improve prompt lebih lanjut?

Gunakan findings ini untuk update `agent_rag.py`!